# Module 16: The Causal Claim, What You Can Defend

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Fifteen modules tried seven designs on one program with a known answer. This
module assembles what survived, and writes the claim.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. Every design this series tried

In [ ]:
dall = f.copy()
dall["lo"] = np.log(dall["n_arrests"])
tb = cell_rate(TRAINED, "before")
ta = cell_rate(TRAINED, "after")

rows = [
    {"design": "before and after",
     "reported": f"{100 * (ta / tb - 1):+.1f}%",
     "the diagnostic that condemns it": "no comparison group"},
    {"design": "regression discontinuity", "reported": "+31%",
     "the diagnostic that condemns it": "two agencies within 0.30 of the cutoff"},
    {"design": "instrumental variables", "reported": "-22.8%",
     "the diagnostic that condemns it": "strongest first stage F is 1.60"},
    {"design": "synthetic control", "reported": "+11.3% to -25.1%",
     "the diagnostic that condemns it": "pre period fit error 25 to 48 percent"},
    {"design": "propensity score", "reported": "none",
     "the diagnostic that condemns it": "4 percent overlap, 2 units survive trimming"},
    {"design": "difference in differences, all five",
     "reported": f"{fit(dall, TRAINED)[0]:+.1f}%",
     "the diagnostic that condemns it": "one agency on its own pre trend"},
    {"design": "difference in differences, checked",
     "reported": f"{fit(d, KEEP)[0]:+.1f}%", "the diagnostic that condemns it": ""},
    {"design": "THE TRUTH", "reported": f"{TRUTH:+.1f}%",
     "the diagnostic that condemns it": ""},
]
pd.DataFrame(rows).set_index("design")

**Four of the seven had no business being run, and each says so in a single
diagnostic that requires no fitting**: a count of units near a cutoff, a first
stage F, a pre period fit error, an overlap range.

Every one of those four still produces a number with a standard error, and
three of them produce a number with the wrong sign or double the truth.

## 3. The checks, assembled

In [ ]:
e, lo, hi, _ = fit(d, KEEP)
z = smf.glm("n_uof ~ C(agency_id)+C(year_month)+settled+phase",
            d.assign(settled=((d["agency_id"].isin(KEEP))
                              & (d["period"] == "after")).astype(float),
                     phase=((d["agency_id"].isin(KEEP))
                            & (d["period"] == "phase")).astype(float)),
            family=sm.families.Poisson(), offset=d["lo"]).fit()
se = z.bse["settled"]

pre = d[d["period"] == "before"].copy()
pre["tr"] = pre["agency_id"].isin(KEEP).astype(float)
zp = smf.glm("n_uof ~ C(agency_id) + yr + tr:yr", pre,
             family=sm.families.Poisson(), offset=pre["lo"]).fit()
k = [x for x in zp.params.index if "yr" in x and "tr" in x][0]
plo, phi = [pct(v) for v in zp.conf_int().loc[k]]

checks = [
    ("estimand", "ATT, weighted by incidents, over 30 settled months"),
    ("estimate", f"{e:+.1f}%  model interval [{lo:+.1f}, {hi:+.1f}]"),
    ("cluster bootstrap interval", "[-20.9, -8.0], the one reported"),
    ("randomisation inference", "p = 0.030 over 400 four agency reassignments"),
    ("smallest detectable effect", f"{100 * (1 - np.exp(-2.80 * se)):.1f}%"),
    ("pre trend difference", f"{pct(zp.params[k]):+.2f}% a year  [{plo:+.2f}, {phi:+.2f}]"),
    ("placebo on arrests", "+0.08%  [-0.94, +1.10]"),
    ("homogeneity across agencies", "chi squared 4.56 on 3 df, p = 0.207"),
    ("breakdown point, hidden trend", "-3.4% a year, inside the pre period interval"),
    ("unobserved selection needed", "3.3 times the pull of the observed controls"),
    ("agencies excluded", "1 treated, pre trend of -12.0% a year"),
    ("months excluded", "1, documented civil unrest"),
]
for lab, val in checks:
    print(f"  {lab:32s} {val}")

## 4. The claim

In [ ]:
print(f"""
ESTIMAND. The average treatment effect on the treated, weighted by incidents,
over the 30 months after full implementation, for the four agencies retained.
The average treatment effect is not identified: agencies were selected on the
pre program outcome and the groups overlap over 4 percent of that variable's
range.

IDENTIFICATION. Parallel trends, no anticipation, no interference, and a
stable outcome definition, with agency and calendar month fixed effects
closing the two backdoor paths in the assumed graph. Four alternative designs
were considered and ruled out on diagnostics stated in the appendix.

ESTIMATE. Use of force ran {abs(e):.1f} percent below the comparison agencies,
cluster bootstrap interval from 20.9 to 8.0 percent below. A randomisation
test over 400 reassignments places the estimate at p = 0.030. The design could
have detected a reduction of {100 * (1 - np.exp(-2.80 * se)):.1f} percent or larger.

WHAT WAS CHECKED. Pre program trends differ by {abs(pct(zp.params[k])):.2f} percent a year,
interval {abs(phi):.2f} above to {abs(plo):.2f} below. Arrests, the exposure measure, were
unaffected. Placebo interventions at eight pre program dates all return
intervals covering zero. Effects do not differ detectably across the four
agencies, though the test would miss a spread below about thirty points.

WHERE IT IS FRAGILE. A hidden trend difference of 3.4 percent a year would
erase the estimate, and the pre period cannot exclude one of 3.31. Partial
identification without the parallel trends assumption bounds the effect only
within 75 percentage points. Unobserved selection would have to be 3.3 times
as influential as everything observed.

WHAT IS NOT CLAIMED. That the program would have this effect at agencies
unlike these five; that the effect arrived abruptly rather than gradually;
that it persists beyond 30 months; that any single agency's estimate is
informative.
""")

Five paragraphs. **The last two are the ones that make the first three worth
reading**, and they are the ones that get cut for length.

## 5. What this series establishes

| | |
|---|---|
| A design produces a number whether or not it is identified | Module 3 |
| Precision is not evidence of identification | Module 3 |
| The estimand depends on the link function | Module 5 |
| The pre trend test would miss the violation that mattered | Module 6 |
| Staggered adoption breaks the estimator two separate ways | Module 7 |
| Four designs fail here, each on one free diagnostic | Modules 8, 10, 11, 12 |
| The model interval is too narrow at eleven clusters | Module 9 |
| The placebo null depends on how many units you pretend to treat | Module 13 |
| Worst case bounds built on the wrong support are wrong, not conservative | Module 14 |
| A subgroup search finds a gap when there is none | Module 15 |

**Ten of these are reasons to report less than the software offers.** That is
what the level is for.

## Exercise

Write the one paragraph version for a chief who has ten seconds, and check
what it had to drop.

In [ ]:
# Fill in the blank, then run.
SHOW = None          # try True

if SHOW:
    print(f"""
    Ten seconds:

      "At the four agencies that adopted the training and were comparable
       beforehand, use of force fell about 13 percent more than at similar
       agencies over the next two and a half years, with a range of 8 to 21
       percent. The agencies were chosen because their rates were already the
       highest in the state, which we accounted for but cannot rule out
       entirely."

    What it dropped, and why that is acceptable:
      the estimand definition, the identification argument, four ruled out
      designs, the detectable effect, and three sensitivity analyses.

    What it kept, and why that is not negotiable:
      the comparison, the range, the time window, and the selection mechanism.
    """)
else:
    print("Set SHOW above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
SHOW = True
```

Four things survive compression to two sentences: **what it was compared to,
the range, the window, and how the agencies were chosen.**

Everything else in the full claim exists to justify those four, and a reader
who trusts the analyst can take them on faith. A reader who does not can ask
for the rest, which is why the rest is written down.

**The selection mechanism is the one that people cut and should not.** It is
the single fact most likely to change a reader's conclusion, it is short, and
omitting it is the difference between a summary and a sales pitch.

</details>

---

## Where this goes

Six series, three levels each, one dataset, one known answer of 12 percent.

The [Time Series series](../../../Time_Series/) covers what the outcome is
doing: trends, seasonality, counts, breaks, and forecasting. This series
covers whether anything was done to it.

Both end in the same place: **an estimate, an interval, a list of what was
checked, and a list of what is still open.** Anything shorter is a claim; that
is a finding.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*